# Bayes Series | Chapter 0: Bayes' Theorem & the Bayesian Worldview

> **Chapter Goal:** Build a deep, working understanding of the Bayesian framework — what makes it different from frequentist thinking, what each component of Bayes' theorem really means, and how beliefs are updated sequentially. By the end you will be able to perform full Bayesian inference on coin flips from scratch and explain the difference between credible intervals and confidence intervals in any interview.

---

## 1. Two Worlds: Frequentist vs Bayesian

### **The Core Disagreement**
In frequentist probability, parameters are **fixed unknowns** — they have true values that we estimate from data. The word "probability" only applies to events that can be repeated many times.

In Bayesian probability, parameters are **random variables** with probability distributions that encode our **degree of belief** about their values. Any uncertainty — including uncertainty about fixed parameters — can be represented as a probability.

| Question | Frequentist Answer | Bayesian Answer |
| :--- | :--- | :--- |
| What is $\theta$? | Fixed, unknown, to be estimated | Random variable with a distribution |
| What is $P(\theta)$? | Meaningless (not a random event) | Prior — our beliefs before data |
| What is inference? | Estimate $\hat\theta$ from data | Update $P(\theta)$ using data |
| What is "95% interval"? | Confidence interval — 95% of intervals from repeated experiments contain $\theta$ | Credible interval — $\theta$ is in here with 95% probability |
| Learning? | One-shot estimation | Sequential belief updating |

### **Why Bayes Matters for ML**
1. **Quantifies uncertainty in predictions** (not just point estimates).
2. **Handles small datasets** — the prior acts as regularization.
3. **Natural handling of missing data** — marginalize over what you don't know.
4. **Principled model comparison** — use marginal likelihood as model score.
5. **Sequential learning** — update beliefs as new data arrives naturally.

---

## 2. Bayes' Theorem — The Engine of Inference

### **The Formula**
$$P(\theta | \mathcal{D}) = \frac{P(\mathcal{D} | \theta) \cdot P(\theta)}{P(\mathcal{D})}$$

$$\underbrace{P(\theta | \mathcal{D})}_{\text{Posterior}} = \frac{\underbrace{P(\mathcal{D}|\theta)}_{\text{Likelihood}} \cdot \underbrace{P(\theta)}_{\text{Prior}}}{\underbrace{P(\mathcal{D})}_{\text{Evidence}}}$$

Or in words: **Posterior $\propto$ Likelihood $\times$ Prior**

### **Each Component — In Depth**

**Prior $P(\theta)$:**
- Encodes beliefs about $\theta$ **before** seeing any data.
- Can be informative (tight distribution near a specific value) or non-informative (uniform — "I have no idea").
- Choice of prior is subjective — this is Bayesian inference's most controversial aspect.
- For large $N$, the prior matters less (data swamps it). For small $N$, the prior is crucial.

**Likelihood $P(\mathcal{D}|\theta)$:**
- How probable is the **observed data** if the parameter were $\theta$?
- This is the **same** quantity as in MLE, but in a different role here: not just one fixed function of $\theta$ to maximize, but one factor in the posterior.
- Large likelihood → data is consistent with this $\theta$ → posterior is high here.
- Note: $P(\mathcal{D}|\theta)$ is a function of $\theta$, not of $\mathcal{D}$.

**Evidence $P(\mathcal{D})$ (Marginal Likelihood):**
- Total probability of observing this data under **all** possible parameter values:
$$P(\mathcal{D}) = \int P(\mathcal{D}|\theta) P(\theta) d\theta$$
- Acts purely as a **normalizing constant** — ensures the posterior integrates to 1.
- Is the same for all values of $\theta$ — doesn't affect which $\theta$ has the highest posterior.
- Is **extremely hard to compute** in general — the main computational challenge of Bayesian inference.
- **For model comparison:** Two models $M_1$ and $M_2$ can be compared by their evidence $P(\mathcal{D}|M_i)$ — the model that assigns higher probability to the observed data wins (Occam's razor is automatic).

**Posterior $P(\theta|\mathcal{D})$:**
- Our updated beliefs about $\theta$ **after** observing data.
- Combines what we believed before (prior) with what the data tells us (likelihood).
- This is the **complete answer** in Bayesian inference — a full distribution, not just a point estimate.
- From the posterior, we can compute: MAP (mode), posterior mean, credible intervals, predictive distributions.

---

## 3. ✍️ Full Example: Coin Flip — Beta-Binomial Conjugate Analysis

**Problem:** You have a possibly biased coin with unknown bias $\theta \in [0,1]$. You flip it $n$ times and observe $k$ heads. What is your belief about $\theta$?

### **Step 1: Choose the Likelihood**
Each flip is $\text{Bernoulli}(\theta)$, $n$ flips give $\text{Binomial}(n, \theta)$:
$$P(k | n, \theta) = \binom{n}{k} \theta^k (1-\theta)^{n-k}$$

### **Step 2: Choose the Prior**
We want a prior on $\theta \in [0,1]$. Use $\text{Beta}(\alpha, \beta)$:
$$P(\theta) = \frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}{B(\alpha,\beta)}$$

Interpretation of hyperparameters: $\alpha$ = prior "pseudo-heads", $\beta$ = prior "pseudo-tails". $\alpha = \beta = 1$ = uniform (no prior information).

### **Step 3: Apply Bayes' Theorem**
$$P(\theta | k, n) \propto P(k|n,\theta) \cdot P(\theta)$$
$$\propto \theta^k(1-\theta)^{n-k} \cdot \theta^{\alpha-1}(1-\theta)^{\beta-1}$$
$$= \theta^{(\alpha + k) - 1}(1-\theta)^{(\beta + n - k) - 1}$$

This is the kernel of a $\text{Beta}(\alpha + k, \beta + n - k)$ distribution! So:
$$\boxed{P(\theta | k, n) = \text{Beta}(\alpha + k, \; \beta + n - k)}$$

**The posterior is Beta with updated counts.** This is the power of conjugate priors — the posterior stays in the same family as the prior.

### **Step 4: Extract Estimates**
| Estimate | Formula | Value (with $\alpha=\beta=1$, $n=10$, $k=7$) |
| :--- | :--- | :--- |
| **MAP** (posterior mode) | $\frac{\alpha+k-1}{\alpha+\beta+n-2}$ | $\frac{7}{10} = 0.700$ |
| **Posterior mean** | $\frac{\alpha+k}{\alpha+\beta+n}$ | $\frac{8}{12} = 0.667$ |
| **MLE** | $k/n$ | $7/10 = 0.700$ |

With uniform prior ($\alpha=\beta=1$): MAP = MLE. Posterior mean is shrunk slightly toward 0.5.

### **The Posterior Mean Formula**
The posterior mean can be written as a **weighted average**:
$$E[\theta|k,n] = \frac{\alpha+k}{\alpha+\beta+n} = \underbrace{\frac{\alpha+\beta}{\alpha+\beta+n}}_{w_\text{prior}} \cdot \underbrace{\frac{\alpha}{\alpha+\beta}}_{\text{prior mean}} + \underbrace{\frac{n}{\alpha+\beta+n}}_{w_\text{data}} \cdot \underbrace{\frac{k}{n}}_{\text{MLE}}$$

- When $n \gg \alpha+\beta$: posterior mean → MLE (data wins)
- When $n \ll \alpha+\beta$: posterior mean → prior mean (prior wins)
- $\alpha+\beta$ = **effective prior sample size** — how many observations the prior is "worth"

---

## 4. Sequential Updating — Learning Incrementally

One of Bayesian inference's most beautiful properties: **you can process data one observation at a time**, and you get the same result as if you processed it all at once.

**Observation 1:** $x_1 = \text{heads}$.
$$P(\theta) = \text{Beta}(1,1) \rightarrow P(\theta|x_1) = \text{Beta}(2, 1)$$

**Observation 2:** $x_2 = \text{tails}$.
$$P(\theta|x_1) = \text{Beta}(2,1) \rightarrow P(\theta|x_1,x_2) = \text{Beta}(2, 2)$$

**Observation 3:** $x_3 = \text{heads}$.
$$P(\theta|x_1,x_2) = \text{Beta}(2,2) \rightarrow P(\theta|x_1,x_2,x_3) = \text{Beta}(3, 2)$$

**Check:** After 2 heads and 1 tail total: $\text{Beta}(1+2, 1+1) = \text{Beta}(3,2)$. ✅ Same answer.

> **Today's posterior = tomorrow's prior.** This makes Bayesian methods naturally suited to online/streaming learning.

**Order doesn't matter** — the posterior after observing $\{H, T, H\}$ is the same as after $\{T, H, H\}$ because the likelihood is exchangeable.

---

## 5. Posterior Predictive Distribution

After seeing data $\mathcal{D}$, what is the probability that the **next** observation $x^*$ takes value $v$?

Frequentist: plug in $\hat\theta_{\text{MLE}}$ and predict: $P(x^* = v) = P(x^*=v|\hat\theta)$. This ignores uncertainty in $\theta$.

Bayesian: **average over all possible $\theta$ values**, weighted by the posterior:
$$P(x^* = v | \mathcal{D}) = \int P(x^*=v|\theta) \cdot P(\theta|\mathcal{D}) \, d\theta = E_{\theta|\mathcal{D}}[P(x^*=v|\theta)]$$

### **For the Coin Flip Example**
$$P(x^* = \text{heads} | \mathcal{D}) = \int_0^1 \theta \cdot \text{Beta}(\theta; \alpha+k, \beta+n-k) \, d\theta = E[\theta|\mathcal{D}] = \frac{\alpha+k}{\alpha+\beta+n}$$

The predictive probability = the posterior mean of $\theta$. This automatically accounts for uncertainty — we don't commit to a single estimate.

**The predictive distribution is always more uncertain than predictions from a single $\hat\theta$.** This is because we marginalize over parameter uncertainty, adding extra variance.

---

## 6. Credible Intervals vs Confidence Intervals

This is one of the most misunderstood topics in statistics — and a very common interview question.

### **Confidence Interval (Frequentist)**
A 95% confidence interval means: **if we repeated the experiment many times**, 95% of the resulting intervals would contain the true (fixed) $\theta$.

❌ **Common mistake:** "There is a 95% probability that $\theta$ is in this interval." This is **wrong** — $\theta$ is fixed; it's either in the interval or not with probability 0 or 1.

### **Credible Interval (Bayesian)**
A 95% credible interval means: **given the observed data**, the probability that $\theta$ lies in this interval is 95%:
$$P(\theta \in [a, b] | \mathcal{D}) = 0.95$$

✅ **This is the intuitive statement** that people actually want to make. The Bayesian framework allows it because $\theta$ is treated as a random variable.

### **Key Differences**
| | Confidence Interval | Credible Interval |
| :--- | :--- | :--- |
| Framework | Frequentist | Bayesian |
| Interprets $\theta$ as | Fixed unknown | Random variable |
| Statement | "95% of intervals cover $\theta$" | "$P(\theta \in I | \mathcal{D}) = 0.95$" |
| Requires prior? | No | Yes |
| Computationally | Often analytical | Often analytical (if conjugate), else MCMC |
| Types | Two-sided, one-sided | Highest Posterior Density (HPD), equal-tail |

---

## 7. Interview Q&A

**Q1: Explain Bayes' theorem in plain English.**
> Bayes' theorem says: posterior ∝ likelihood × prior. It's how we update our beliefs (prior) about a parameter in light of new evidence (data via the likelihood). The posterior is our updated belief distribution after observing the data.

**Q2: What is the prior and how do you choose it?**
> The prior $P(\theta)$ encodes our beliefs about $\theta$ before seeing data. Choosing it is the subjective part of Bayesian inference. We can use informative priors (based on domain knowledge), weakly informative priors (regularizing but not strongly biased), or non-informative/uniform priors (maximum data influence). For large $N$, the prior matters less; for small $N$, it's crucial.

**Q3: What is the evidence/marginal likelihood, and why is it useful?**
> $P(\mathcal{D}) = \int P(\mathcal{D}|\theta)P(\theta)d\theta$ is the total probability of the data under all parameter values. As a normalizer, it ensures the posterior integrates to 1. For model comparison, it's the score assigned to the entire model $M$: $P(\mathcal{D}|M)$. Models that explain the data well without overfitting have high marginal likelihood — Bayesian model selection automatically embodies Occam's razor.

**Q4: What is the posterior predictive distribution?**
> $P(x^*|\mathcal{D}) = \int P(x^*|\theta) P(\theta|\mathcal{D}) d\theta$. It predicts new outcomes by averaging predictions over all possible parameter values, weighted by the posterior. It accounts for parameter uncertainty, giving more honest (wider) predictions than plugging in a point estimate $\hat\theta$.

**Q5: What is a credible interval and how does it differ from a confidence interval?**
> A 95% credible interval $[a,b]$ satisfies $P(\theta \in [a,b] | \mathcal{D}) = 0.95$ — a direct probability statement about $\theta$. A 95% confidence interval means 95% of intervals constructed this way across repeated experiments would contain the true $\theta$ — a statement about the procedure, not about this specific $\theta$. Credible intervals give the intuitive interpretation that most people incorrectly attribute to confidence intervals.

**Q6: What is "conjugate prior" and why is it useful?**
> A conjugate prior gives a posterior in the same distributional family as the prior. Example: Beta prior + Binomial likelihood → Beta posterior. This enables closed-form Bayesian inference — no numerical integration or MCMC needed. The posterior parameters are simple functions of the prior parameters and data statistics.

**Q7: How does Bayesian inference handle overfitting?**
> By marginalizing over parameters rather than fixing them at point estimates, Bayesian inference automatically penalizes model complexity. More complex models have a larger parameter space, and the marginal likelihood integrates over all of it — most of that space has low likelihood, so complex models tend to have lower evidence unless the extra complexity is genuinely supported by data. This is automatic regularization.

**Q8: What happens to the posterior as $N \to \infty$?**
> The likelihood term dominates the prior — the posterior concentrates sharply around the MLE (Bernstein-von Mises theorem). The posterior becomes Gaussian (by CLT-like arguments) with mean → MLE and variance → $1/(N \cdot \mathcal{I}(\theta))$ where $\mathcal{I}$ is the Fisher information. All reasonable priors give the same posterior in this limit.

---

In [ ]:
# ─── CELL 1: Full Bayesian Analysis — Coin Flip Sequential Updating ────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
true_theta = 0.70  # true but unknown bias
n_total = 50
flips = np.random.binomial(1, true_theta, n_total)  # 0=tails, 1=heads

# Different starting priors
priors = [
    (1, 1,   'Uniform Beta(1,1)',    'steelblue'),
    (5, 5,   'Symmetric Beta(5,5)',  'darkorange'),
    (10, 2,  'Informative Beta(10,2) — believes θ≈0.83', 'forestgreen'),
]
x = np.linspace(0.001, 0.999, 500)

# Show updating at several checkpoints
checkpoints = [0, 1, 5, 15, 50]
fig, axes = plt.subplots(len(priors), len(checkpoints), figsize=(20, 11), sharex=True)

for row, (a0, b0, prior_label, color) in enumerate(priors):
    for col, n in enumerate(checkpoints):
        k = flips[:n].sum() if n > 0 else 0
        a_post = a0 + k
        b_post = b0 + (n - k)
        ax = axes[row][col]

        # Current posterior
        pdf = stats.beta.pdf(x, a_post, b_post)
        ax.plot(x, pdf, color=color, lw=2.5)
        ax.fill_between(x, pdf, alpha=0.25, color=color)

        # 95% Credible Interval (equal-tail)
        ci_lo = stats.beta.ppf(0.025, a_post, b_post)
        ci_hi = stats.beta.ppf(0.975, a_post, b_post)
        ax.axvspan(ci_lo, ci_hi, alpha=0.15, color=color, label='95% CI')
        ax.axvline(true_theta, color='red', ls='--', lw=1.5)

        post_mean = a_post / (a_post + b_post)
        ax.axvline(post_mean, color='purple', ls='-', lw=1.5, alpha=0.7)

        title = f'n={n}\nk={k}, [CI: {ci_lo:.2f},{ci_hi:.2f}]'
        ax.set_title(title, fontsize=8)
        ax.set_xlim(0, 1); ax.grid(alpha=0.3)
        if col == 0: ax.set_ylabel(f'{prior_label}\n(rows)', fontsize=7)

plt.suptitle(f'Bayesian Sequential Updating — Coin Flip (True θ={true_theta})\n'
             'Red=true θ, Purple=posterior mean, Shaded=95% credible interval\n'
             'As n→50, all priors converge to the same posterior', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: Posterior Predictive vs Plug-in Prediction ───────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(0)
true_theta = 0.7
x_theta = np.linspace(0.001, 0.999, 400)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Show scenario: n=3 flips → 2 heads (very small data)
k, n = 2, 3
a0, b0 = 1, 1  # uniform prior
a_post, b_post = a0+k, b0+n-k

# Left: the posterior
ax = axes[0]
pdf = stats.beta.pdf(x_theta, a_post, b_post)
ax.plot(x_theta, pdf, lw=2.5, color='steelblue', label=f'Posterior Beta({a_post},{b_post})')
ax.fill_between(x_theta, pdf, alpha=0.3, color='steelblue')
mle = k/n
post_mean = a_post/(a_post+b_post)
ax.axvline(mle, color='red', ls='--', lw=2, label=f'MLE θ = {mle:.2f}')
ax.axvline(post_mean, color='purple', ls='-', lw=2, label=f'Post. mean θ = {post_mean:.2f}')
ax.axvline(true_theta, color='green', ls=':', lw=1.5, label=f'True θ = {true_theta}')
ax.set_title(f'Posterior after n={n} flips, k={k} heads\n(Uniform prior)', fontsize=11)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Middle: predictive probability of next heads — posterior predictive vs plug-in
ax = axes[1]
N_future = 20  # ask: what are probabilities of 0,1,...,N_future heads in next N_future flips?
k_vals = np.arange(0, N_future+1)

# Plug-in predictive: use MLE
plugin_pred = stats.binom.pmf(k_vals, N_future, mle)

# Posterior predictive: Beta-Binomial distribution (analytically integrated)
# P(k | n_new, data) = BetaBin(k; n_new, a_post, b_post)
from scipy.special import betaln
def betabinom_pmf(k, n_new, a, b):
    from scipy.special import comb
    log_p = (np.log(comb(n_new, k, exact=False))
             + betaln(k + a, n_new - k + b)
             - betaln(a, b))
    return np.exp(log_p)
post_pred = np.array([betabinom_pmf(k_val, N_future, a_post, b_post) for k_val in k_vals])

ax.bar(k_vals - 0.2, plugin_pred, 0.35, alpha=0.7, color='red', label=f'Plug-in MLE (θ={mle:.2f})')
ax.bar(k_vals + 0.2, post_pred, 0.35, alpha=0.7, color='steelblue', label='Posterior predictive\n(accounts for uncertainty)')
ax.set_xlabel(f'Heads in next {N_future} flips'); ax.set_ylabel('Probability')
ax.set_title(f'Predictive Distribution for next {N_future} flips\n(n={n}, k={k} — very little data)', fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

# Right: width of 95% credible interval as N grows
ax = axes[2]
N_vals = range(1, 101)
ci_widths = []
for n_obs in N_vals:
    k_obs = int(round(0.7 * n_obs))
    a_p, b_p = 1 + k_obs, 1 + n_obs - k_obs
    lo = stats.beta.ppf(0.025, a_p, b_p)
    hi = stats.beta.ppf(0.975, a_p, b_p)
    ci_widths.append(hi - lo)

ax.plot(N_vals, ci_widths, color='steelblue', lw=2)
ax.fill_between(N_vals, ci_widths, alpha=0.2, color='steelblue')
ax.set_xlabel('Number of observations N'); ax.set_ylabel('Width of 95% credible interval')
ax.set_title('Uncertainty Shrinks with Data\n(95% credible interval width vs N)', fontsize=11)
ax.grid(alpha=0.3)

# Add approximate bound: width ~ 1/sqrt(N)
ax.plot(N_vals, 2.0/np.sqrt(N_vals), 'r--', lw=1.5, label='~2/√N (approximate)', alpha=0.7)
ax.legend(fontsize=9)

plt.suptitle('Posterior Predictive Distribution: Accounting for Parameter Uncertainty',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nWith only n={n} observations:")
print(f"  Plug-in var in next {N_future} flips (using MLE): {N_future*mle*(1-mle):.2f}")
print(f"  Posterior predictive var (accounts for θ uncertainty): higher (wider distribution above)")
print(f"  Posterior predictive mean = {(post_pred * k_vals).sum():.2f} (should ≈ {N_future*post_mean:.2f})")